# Lab 22 - T4 Full Run With Bonus

Notebook nay duoc tao de chay tren Google Colab T4 voi folder da upload san tai:

`/content/drive/MyDrive/AI_IN_ACTION/2A202600883-MaiHanhPham-Day22-Track3-DPO-Alignment-Lab`

Thu tu chay:

1. Mount Drive va vao dung folder lab
2. Setup dependencies
3. Smoke check
4. Core NB1 -> NB4
5. Bonus NB5, NB6, beta-sweep
6. Verify submission

Luu y: T4 co the chay core. Bonus `deploy`, `bench`, va dac biet `beta-sweep` co the rat lau hoac het VRAM. Neu sap het thoi gian, uu tien core + verify truoc.

## 0. Mount Drive And Enter Lab Folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

LAB_DIR = '/content/drive/MyDrive/AI_IN_ACTION/2A202600883-MaiHanhPham-Day22-Track3-DPO-Alignment-Lab'
%cd $LAB_DIR

!pwd
!ls -la | head

## 1. Confirm GPU - Screenshot 01

Chup cell nay lam `01-setup-gpu.png`.

In [ ]:
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name)
    print('VRAM GB:', round(props.total_memory / 1024**3, 2))

## 2. Set Environment Variables

Mac dinh dung T4 tier. Neu co OpenAI/Anthropic key de auto judge, dien vao cell ben duoi truoc khi chay NB4. Neu khong co key, NB4 se fallback manual rubric.

In [ ]:
import os

os.environ['COMPUTE_TIER'] = 'T4'
os.environ['PYTHONUTF8'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Optional judge keys. Uncomment and paste if you have keys.
# os.environ['OPENAI_API_KEY'] = 'sk-...'
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
# os.environ['JUDGE_MODEL'] = 'gpt-4o-mini'

print('COMPUTE_TIER =', os.environ['COMPUTE_TIER'])

## 3. Full Environment And Library Install

Cum cell nay cai day du moi truong va thu vien can thiet cho core + bonus. Chay cell nay truoc setup de tranh thieu package tren Colab runtime moi.

In [ ]:
import sys, platform, os

print('Python:', sys.version)
print('Executable:', sys.executable)
print('Platform:', platform.platform())
print('Working dir:', os.getcwd())

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA:', torch.version.cuda)
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
%%time
# System packages for notebook execution, native wheels, llama-cpp/GGUF bonus, plotting, and FFmpeg libs.
!apt-get update -qq
!apt-get install -y -qq build-essential cmake ninja-build git-lfs zip unzip ffmpeg > /tmp/lab22_apt.log
!git lfs install
!echo 'System packages installed.'


In [ ]:
%%time
# Python packaging basics. Keep Colab's own notebook kernel packages stable.
!python -m pip install -U pip setuptools wheel packaging

# Core lab dependencies. requirements.txt pins transformers<4.57 to avoid the torchcodec import issue on Colab torch 2.10/cu128.
!python -m pip install -r requirements.txt

# Defensive fix for Colab images where transformers/torchcodec get resolved incompatibly.
!python -m pip uninstall -y torchcodec || true
!python -m pip install 'transformers>=4.46,<4.57'

# Only install tools needed by Makefile notebook execution; do not upgrade ipykernel/jupyter-server/notebook.
!python -m pip install -U 'jupytext>=1.16,<2.0' 'nbconvert>=7,<8'


In [ ]:
# Optional BigGPU extras. Leave RUN_BIGGPU_EXTRAS=False on T4 unless you specifically need vLLM.
RUN_BIGGPU_EXTRAS = False

if RUN_BIGGPU_EXTRAS:
    !python -m pip install -r requirements-biggpu.txt
else:
    print('Skipping requirements-biggpu.txt on T4.')

In [ ]:
# Import check for the libraries used by core + bonus notebooks.
mods = [
    'torch', 'unsloth', 'transformers', 'trl', 'peft', 'accelerate', 'bitsandbytes',
    'datasets', 'matplotlib', 'pandas', 'pyarrow', 'jupytext',
    'openai', 'anthropic', 'llama_cpp', 'lm_eval'
]

import importlib.util
missing = []
for mod in mods:
    ok = importlib.util.find_spec(mod) is not None
    print(f'{mod:16s}', 'OK' if ok else 'MISSING')
    if not ok:
        missing.append(mod)

import transformers
print('transformers version:', transformers.__version__)

if missing:
    raise RuntimeError('Missing packages: ' + ', '.join(missing))

print('All required Python packages are importable.')


## 4. Setup Colab Dependencies

Cell nay co the mat vai phut. Neu Colab yeu cau restart runtime sau khi cai package, restart roi chay lai tu cell 0 den cell nay.

In [ ]:
%%time
!bash setup-colab.sh

## 5. Smoke Check

In [ ]:
!python scripts/verify.py --smoke

# Core Pipeline

## 6. NB1 - SFT Mini

Sau khi chay xong, artifact can co:

- `adapters/sft-mini/adapter_config.json`
- `submission/screenshots/02-sft-loss.png`

Chup loss curve lam `02-sft-loss.png` neu can.

In [ ]:
%%time
import subprocess
subprocess.run(['make', 'sft'], check=True)


In [ ]:
!test -f adapters/sft-mini/adapter_config.json && echo 'OK: SFT adapter exists'
!ls -lh adapters/sft-mini | head
!ls -lh submission/screenshots | sed -n '1,20p'

## 7. NB2 - Preference Data

Sau khi chay xong, artifact can co:

- `data/pref/train.parquet`
- `data/pref/eval.parquet`
- Output in ra 3 preference examples

In [ ]:
%%time
import subprocess
subprocess.run(['make', 'data'], check=True)


In [ ]:
!test -f data/pref/train.parquet && echo 'OK: preference train parquet exists'
!ls -lh data/pref

## 8. NB3 - DPO Train

Sau khi chay xong, artifact can co:

- `adapters/dpo/adapter_config.json`
- `adapters/dpo/dpo_metrics.json`
- `submission/screenshots/03-dpo-reward-curves.png`

Chup reward curves lam `03-dpo-reward-curves.png`. Reflection section 3 phai noi rieng ve chosen reward va rejected reward.

In [ ]:
%%time
import subprocess
subprocess.run(['make', 'dpo'], check=True)


In [ ]:
!test -f adapters/dpo/adapter_config.json && echo 'OK: DPO adapter exists'
!test -f adapters/dpo/dpo_metrics.json && echo 'OK: DPO metrics exists'
!cat adapters/dpo/dpo_metrics.json
!ls -lh submission/screenshots | sed -n '1,30p'

## 9. NB4 - Compare And Eval

Sau khi chay xong, artifact can co:

- `data/eval/side_by_side.jsonl`
- `data/eval/judge_results.json`
- `submission/screenshots/04-side-by-side-table.png`

Chup table lam `04-side-by-side-table.png`. Neu khong co API key, judge output se la manual placeholder; ban can tu dien manual judgment trong reflection.

In [ ]:
%%time
import subprocess
subprocess.run(['make', 'eval'], check=True)


In [ ]:
!test -f data/eval/side_by_side.jsonl && echo 'OK: side_by_side exists'
!test -f data/eval/judge_results.json && echo 'OK: judge_results exists'
!cat data/eval/judge_results.json
!ls -lh data/eval
!ls -lh submission/screenshots | sed -n '1,40p'

# Bonus Tasks

Chay cac cell duoi neu con thoi gian. Tren T4, bonus co the lau va de gap loi package/VRAM hon core.

## 10. Bonus NB5 - Merge And GGUF Deploy

Artifact bonus:

- `gguf/*.gguf`
- `data/eval/deploy_meta.json`
- Screenshot `06-gguf-smoke.png` tu output llama-cpp-python

In [ ]:
%%time
import subprocess
try:
    subprocess.run(['make', 'deploy'], check=True)
except subprocess.CalledProcessError as exc:
    print('NB5 deploy failed or skipped; core grade can still pass:', exc)


In [ ]:
!ls -lh gguf || true
!test -f data/eval/deploy_meta.json && cat data/eval/deploy_meta.json || true

## 11. Bonus NB6 - Benchmark

Artifact bonus:

- `data/eval/benchmark_results.json`
- `submission/screenshots/07-benchmark-comparison.png`

Neu khong co API key, AlpacaEval-lite se bi skip, nhung IFEval/GSM8K/MMLU van co the chay.

In [ ]:
%%time
import subprocess
try:
    subprocess.run(['make', 'bench'], check=True)
except subprocess.CalledProcessError as exc:
    print('NB6 benchmark failed or skipped; core grade can still pass:', exc)


In [ ]:
!test -f data/eval/benchmark_results.json && cat data/eval/benchmark_results.json || true
!ls -lh submission/screenshots | sed -n '1,60p'

## 12. Bonus Beta Sweep

Canh bao: cell nay train DPO them 3 lan voi beta 0.05, 0.1, 0.5. Tren T4 co the mat rat lau. Neu khong chay, reflection section 5 co the viet hypothesis, khong mat diem core.

In [ ]:
%%time
import subprocess
try:
    subprocess.run(['make', 'beta-sweep'], check=True)
except subprocess.CalledProcessError as exc:
    print('beta-sweep failed or skipped; core grade can still pass:', exc)


In [ ]:
!ls -lh adapters | sed -n '1,80p'
!ls -lh submission/screenshots | sed -n '1,80p'

# Final Verify

## 13. Fill Reflection Before Final Verify

Mo `submission/REFLECTION.md` va dien so lieu that tu cac artifact:

- `adapters/dpo/dpo_metrics.json`
- `data/eval/judge_results.json`
- optional `data/eval/benchmark_results.json`

Neu chua chup screenshot, chua can verify pass ngay. Sau khi them screenshot va dien reflection thi chay cell verify ben duoi.

In [ ]:
!python scripts/verify.py

## 14. Package Helpful Outputs

Cell nay tao zip artifacts nhe de download neu can. Khong zip model weights lon.

In [ ]:
!mkdir -p submission/export
!zip -r submission/export/lab22-evidence.zip submission/REFLECTION.md submission/screenshots data/eval adapters/dpo/dpo_metrics.json adapters/sft-mini/adapter_config.json adapters/dpo/adapter_config.json -x '*.bin' '*.safetensors' '*.gguf' || true
!ls -lh submission/export